# 01B Duplicate Checks

This notebook focuses only on duplicate-related data quality issues.

The goal is to check whether the dataset contains repeated records, repeated IDs, repeated track names, or different versions of the same song, such as remasters, live versions, acoustic versions, remixes, or radio edits.

In [1]:
from pathlib import Path
import sys
import duckdb
import pandas as pd
import numpy as np
import re

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 120)

PROJECT_ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from config.paths import DUCKDB_PATH

con = duckdb.connect(DUCKDB_PATH)

tracks_df = con.execute("SELECT * FROM tracks").df()
artists_df = con.execute("SELECT * FROM artists").df()
albums_df = con.execute("SELECT * FROM albums").df()
audio_features_df = con.execute("SELECT * FROM audio_features").df()
lyrics_features_df = con.execute("SELECT * FROM lyrics_features").df()

print("Data loaded successfully.")
print(f"Tracks: {len(tracks_df):,}")
print(f"Artists: {len(artists_df):,}")
print(f"Albums: {len(albums_df):,}")
print(f"Audio features: {len(audio_features_df):,}")
print(f"Lyrics features: {len(lyrics_features_df):,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Data loaded successfully.
Tracks: 101,939
Artists: 56,129
Albums: 75,511
Audio features: 101,909
Lyrics features: 94,954


## Exact Row Duplicates

First, we check whether any table contains fully duplicated rows.

This is the most basic duplicate check. However, it only catches records where every column is exactly the same.

In [2]:
tables = {
    "tracks": tracks_df,
    "artists": artists_df,
    "albums": albums_df,
    "audio_features": audio_features_df,
    "lyrics_features": lyrics_features_df
}

exact_duplicate_results = []

for table_name, df in tables.items():
    exact_duplicate_results.append({
        "table": table_name,
        "rows": len(df),
        "exact_duplicate_rows": df.duplicated().sum(),
        "exact_duplicate_pct": round(df.duplicated().mean() * 100, 2)
    })

exact_duplicate_df = pd.DataFrame(exact_duplicate_results)
exact_duplicate_df

,table,rows,exact_duplicate_rows,exact_duplicate_pct
0,tracks,101939,0,0.0
1,artists,56129,0,0.0
2,albums,75511,0,0.0
3,audio_features,101909,0,0.0
4,lyrics_features,94954,0,0.0


## Duplicate Identifier Check

Next, we check whether the main identifier columns are unique.

This is important because duplicate IDs could cause incorrect joins or repeated records in later analysis.

In [3]:
id_checks = [
    ("tracks", tracks_df, "id"),
    ("artists", artists_df, "id"),
    ("albums", albums_df, "id"),
    ("audio_features", audio_features_df, "track_id"),
    ("lyrics_features", lyrics_features_df, "track_id")
]

id_duplicate_results = []

for table_name, df, id_col in id_checks:
    id_duplicate_results.append({
        "table": table_name,
        "id_column": id_col,
        "rows": len(df),
        "unique_ids": df[id_col].nunique(),
        "duplicate_ids": len(df) - df[id_col].nunique()
    })

id_duplicate_df = pd.DataFrame(id_duplicate_results)
id_duplicate_df

,table,id_column,rows,unique_ids,duplicate_ids
0,tracks,id,101939,101939,0
1,artists,id,56129,56129,0
2,albums,id,75511,75511,0
3,audio_features,track_id,101909,101909,0
4,lyrics_features,track_id,94954,94954,0


## Duplicate Track Names

Spotify track IDs can be unique even when the same song title appears several times.

This check identifies track names that occur more than once in the dataset.

In [4]:
duplicate_track_names = (
    tracks_df
    .groupby("name")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

duplicate_track_names_top = duplicate_track_names[duplicate_track_names["count"] > 1]

print(f"Track names appearing more than once: {len(duplicate_track_names_top):,}")
print(f"Total rows involved in repeated track names: {duplicate_track_names_top['count'].sum():,}")

duplicate_track_names_top.head(30)

Track names appearing more than once: 8,646
Total rows involved in repeated track names: 25,033


,name,count
31689,Home,45
82339,You,39
74187,Time,28
15803,Crazy,26
10823,Breathe,26
4250,Alive,25
20356,Drive,25
31549,Hold On,23
81333,With You,23
20265,Dreams,23


## Normalized Track Names

Track names may differ only because of capitalization, punctuation, or extra spaces.

To detect more realistic duplicates, we create a simplified version of the track name.

In [5]:
def normalize_text(text):
    if pd.isna(text):
        return ""
    
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\w\s]", "", text)
    
    return text

tracks_df["name_clean"] = tracks_df["name"].apply(normalize_text)

duplicate_clean_names = (
    tracks_df
    .groupby("name_clean")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

duplicate_clean_names_top = duplicate_clean_names[duplicate_clean_names["count"] > 1]

print(f"Normalized track names appearing more than once: {len(duplicate_clean_names_top):,}")
print(f"Total rows involved in repeated normalized names: {duplicate_clean_names_top['count'].sum():,}")

duplicate_clean_names_top.head(30)

Normalized track names appearing more than once: 9,262
Total rows involved in repeated normalized names: 27,200


,name_clean,count
31236,home,47
81477,you,42
73374,time,30
10556,breathe,28
15489,crazy,26
3995,alive,26
20007,drive,25
19914,dreams,25
60403,run,24
80479,with you,24


## Version Indicator Detection

Many Spotify duplicates are not exact duplicates.  
The same song may appear as a remaster, live version, acoustic version, remix, radio edit, deluxe version, or instrumental version.

We therefore search track names for common version indicators.

In [6]:
version_patterns = {
    "remaster": r"remaster|remastered",
    "live": r"\blive\b|live at|live from|live in",
    "acoustic": r"acoustic",
    "remix": r"remix|rework|edit",
    "radio_edit": r"radio edit",
    "deluxe": r"deluxe",
    "instrumental": r"instrumental",
    "karaoke": r"karaoke",
    "demo": r"\bdemo\b",
    "anniversary": r"anniversary",
    "sped_up_slowed": r"sped up|slowed|nightcore",
    "explicit_clean": r"clean version|explicit version"
}

version_summary = []

for label, pattern in version_patterns.items():
    mask = tracks_df["name"].str.lower().str.contains(pattern, regex=True, na=False)
    
    version_summary.append({
        "version_type": label,
        "track_count": mask.sum(),
        "share_pct": round(mask.mean() * 100, 2)
    })

version_summary_df = (
    pd.DataFrame(version_summary)
    .sort_values("track_count", ascending=False)
)

version_summary_df

,version_type,track_count,share_pct
3,remix,3364,3.30
0,remaster,1003,0.98
1,live,830,0.81
4,radio_edit,699,0.69
2,acoustic,621,0.61
6,instrumental,182,0.18
8,demo,32,0.03
9,anniversary,9,0.01
7,karaoke,5,0.00
5,deluxe,1,0.00


In [7]:
version_example_mask = tracks_df["name"].str.lower().str.contains(
    "|".join(version_patterns.values()),
    regex=True,
    na=False
)

tracks_df.loc[
    version_example_mask,
    ["id", "name", "album_id", "artists_id", "popularity"]
].head(50)

,id,name,album_id,artists_id,popularity
54,5J0aH9sTGhYTW6M2vuwk5l,It Would Never Have Worked - Live,6hkfojKabPHetxgdYAAfWP,['14Ccdm516mXvMqVywHIcj7'],23.0
77,4lMG05ZPmd021uMdkU1xUO,Ocean (feat. Alexandra Stan) - Radio Edit,7zsrou182C71CynS7Vl4yy,"['7mz4XMo31wsaANzVn7xdd1', '0BmLNz4nSLfoWYW1cY...",33.0
82,0TmjbzJHDDe6THTGmanJWO,Symphony (feat. Zara Larsson) - R3hab Remix,4niCexwO1zY6BsaOIsOck8,"['6MDME20pz9RveH9rEXvrOM', '1Xylc3o4UrD53lo9Cv...",29.0
99,0ZBPWoRkfZ8SLcDJaFnkUu,Passion - Naked Edit,4F74H7yribKYcseNcUvgv5,['6wbsiIvg0rsbL9JlLAH9GA'],46.0
106,4YouyrGDNnMiuTpYiUklOc,The Undisputed Champs (feat. Q-Tip of A Tribe ...,2XYtkhFpj90gSdbdmk0Wur,"['0YsLR3SQd5QTXAhGIGX7cl', '3ZotbHeyVQKxQCPDJu...",36.0
111,56TB6Wsrl6MgWBkKWSnjd7,Is There Anybody out There? - Radio Edit,5MOG4UkYIhHJ3G8pFcrXRj,['1ezo830nMkWmjhmpjr6vuF'],38.0
150,6Hey64wSB5r98RMd4ozzsy,Keeper of the Flame - Radio Edit,4I81zDK5QUqbxgNYJaj0jm,['66lH4jAE7pqPlOlzUKbwA0'],50.0
176,2iejTMy9XZ8Gaae0aQ2yl0,If I Ain't Got You - Radio Edit,1gAM7M4rBwEbSPeAQR2nx1,['3DiDSECUqqY1AuBP8qtaIa'],41.0
198,6L6cHKXzwbSZtFxW8xZpui,Freed from Desire (feat. Indiiana) - DNF Remix,0DRnn83wzZWgVz23W4uCUl,"['3nmaO18tcMzfrrR7sdJHnH', '6oHCxanXnGoT4ntOtX...",45.0
218,2fLfzTd72ei3pw7mHBrlF3,Somebody I'm Not - Cahill Edit,1q9AdQSFTO3SKqj4KbFZee,"['4ehtJnVumNf6xzSCDk8aLB', '16EmDsIYHYtxgSNfjE...",52.0


## Potential Duplicate Songs by Name and Artist

The strongest duplicate signal is when the same artist has multiple tracks with the same or very similar track name.

This can happen because of re-releases, remasters, deluxe editions, compilation albums, live versions, or duplicate playlist entries.

In [8]:
tracks_artist_duplicates = (
    tracks_df
    .groupby(["artists_id", "name_clean"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

potential_song_duplicates = tracks_artist_duplicates[
    tracks_artist_duplicates["count"] > 1
]

print(f"Potential duplicate song groups: {len(potential_song_duplicates):,}")
print(f"Total rows involved: {potential_song_duplicates['count'].sum():,}")

potential_song_duplicates.head(30)

Potential duplicate song groups: 3,887
Total rows involved: 8,548


,artists_id,name_clean,count
6011,['0SfsnGyD8FpIN4U4WCkBZ5'],blah blah blah,9
25711,['2424G4yH7tJBPlbAoiVmc3'],nouns 1,8
52122,['49FYG2DSZqIVpCtoir2Upw'],expressions 1,8
21674,['1l6d0RIxTL3JytlLGvWzYe'],benjamin blümchen gutenachtgeschichten folge ...,8
52178,['49FYG2DSZqIVpCtoir2Upw'],verbs 2,8
52177,['49FYG2DSZqIVpCtoir2Upw'],verbs 1,8
52123,['49FYG2DSZqIVpCtoir2Upw'],expressions 2,8
52128,['49FYG2DSZqIVpCtoir2Upw'],nouns 1,8
52129,['49FYG2DSZqIVpCtoir2Upw'],nouns 2,8
25714,['2424G4yH7tJBPlbAoiVmc3'],nouns 4,7


In [9]:
example_duplicate_groups = potential_song_duplicates.head(10)

duplicate_examples = tracks_df.merge(
    example_duplicate_groups[["artists_id", "name_clean"]],
    on=["artists_id", "name_clean"],
    how="inner"
)

duplicate_examples[
    ["id", "name", "artists_id", "album_id", "popularity", "duration_ms"]
].sort_values(["artists_id", "name"])

,id,name,artists_id,album_id,popularity,duration_ms
1,5TFVfE1zNbyRG18NKeSL8z,Blah Blah Blah,['0SfsnGyD8FpIN4U4WCkBZ5'],0tFAEse8MKYy0svkf1DOSw,28.0,183519.0
2,4ECA0jaumK20cc7vCqoYAA,Blah Blah Blah,['0SfsnGyD8FpIN4U4WCkBZ5'],1pqNyJwJ2IZ1xvecqDrsfw,56.0,183519.0
6,0WQf8kcrI1yb6l2vpKjsOI,Blah Blah Blah,['0SfsnGyD8FpIN4U4WCkBZ5'],37YRnekRlivoePTsv9pJDp,69.0,183519.0
22,7JoVRy6b5AqggrBcSQ27wV,Blah Blah Blah,['0SfsnGyD8FpIN4U4WCkBZ5'],6NkuIBhMPuDsskoAPG4w8y,58.0,183519.0
34,1m1nkTDqTntuG9HaOo4Jn7,Blah Blah Blah,['0SfsnGyD8FpIN4U4WCkBZ5'],6UAdSODOUh9cbejLd6f5Ts,70.0,183519.0
49,7c97wKkJEAbldEQ6FV3ZqW,Blah Blah Blah,['0SfsnGyD8FpIN4U4WCkBZ5'],0mvmtikiHaNKd3ZKGAMKgR,43.0,183519.0
50,2xkrujtSjZz7EKAYGbIIzH,Blah Blah Blah,['0SfsnGyD8FpIN4U4WCkBZ5'],1oqv6oabMXbNpgZ6gmjAIN,69.0,183519.0
67,0tp6mwUt99QnfY95T30Lqj,Blah Blah Blah,['0SfsnGyD8FpIN4U4WCkBZ5'],0ZHwstiHnsa3b9RepMKFfb,67.0,183519.0
69,7sJjZqH6GMCdkNiOTru6wc,Blah Blah Blah,['0SfsnGyD8FpIN4U4WCkBZ5'],6SEVFEJ2wAhCbCdEz5v2dg,22.0,183519.0
15,38hpP75U1Kr85KLHJ9zNQv,Benjamin Blümchen Gute-Nacht-Geschichten - Fol...,['1l6d0RIxTL3JytlLGvWzYe'],2q6onixYvSOLx6rTInOGe1,13.0,299253.0


## Duration-Based Duplicate Check

Some duplicated songs may have slightly different titles but very similar durations.

Here we check repeated artist-title combinations and compare their track durations.

In [10]:
duplicate_duration_summary = (
    tracks_df
    .merge(
        potential_song_duplicates[["artists_id", "name_clean"]],
        on=["artists_id", "name_clean"],
        how="inner"
    )
    .groupby(["artists_id", "name_clean"])
    .agg(
        track_count=("id", "count"),
        min_duration_ms=("duration_ms", "min"),
        max_duration_ms=("duration_ms", "max"),
        mean_popularity=("popularity", "mean")
    )
    .reset_index()
)

duplicate_duration_summary["duration_range_sec"] = (
    (duplicate_duration_summary["max_duration_ms"] - duplicate_duration_summary["min_duration_ms"]) / 1000
).round(2)

duplicate_duration_summary.sort_values("track_count", ascending=False).head(30)

,artists_id,name_clean,track_count,min_duration_ms,max_duration_ms,mean_popularity,duration_range_sec
238,['0SfsnGyD8FpIN4U4WCkBZ5'],blah blah blah,9,183519.0,183519.0,53.555556,0.00
2085,['49FYG2DSZqIVpCtoir2Upw'],nouns 2,8,510427.0,1630680.0,8.750000,1120.25
2080,['49FYG2DSZqIVpCtoir2Upw'],expressions 1,8,622533.0,1197320.0,9.875000,574.79
857,['1l6d0RIxTL3JytlLGvWzYe'],benjamin blümchen gutenachtgeschichten folge ...,8,293400.0,340880.0,13.375000,47.48
2091,['49FYG2DSZqIVpCtoir2Upw'],verbs 2,8,603133.0,1320853.0,6.375000,717.72
2081,['49FYG2DSZqIVpCtoir2Upw'],expressions 2,8,591320.0,1372533.0,7.750000,781.21
2090,['49FYG2DSZqIVpCtoir2Upw'],verbs 1,8,618720.0,1357453.0,8.000000,738.73
1031,['2424G4yH7tJBPlbAoiVmc3'],nouns 1,8,638880.0,2393173.0,6.875000,1754.29
2084,['49FYG2DSZqIVpCtoir2Upw'],nouns 1,8,572840.0,1323960.0,11.375000,751.12
1038,['2424G4yH7tJBPlbAoiVmc3'],verbs 4,7,234133.0,729813.0,3.714286,495.68


## Last Checks

In [11]:
# tracks longer than 15 minutes

long_tracks = tracks_df[
    tracks_df["duration_ms"] > 15 * 60 * 1000
]

print(len(long_tracks))

long_tracks[
    ["name", "duration_ms", "popularity"]
].sort_values("duration_ms", ascending=False).head(100)

1108


,name,duration_ms,popularity
48041,A Guide to Men - Helen Rowland,5505831.0,17.0
76678,Introductory Lecture to Buddhism,4811520.0,12.0
95353,Stadium Hotel,4770973.0,9.0
36904,Thanksgiving,4694040.0,13.0
44351,The Letters & Journals Of Lord Nelson - Part 1,4667397.0,5.0
7190,The Adventure of the Speckled Band,4508318.0,10.0
51104,"Thunderstorm, Rain, Ocean Waves - Relaxing Sou...",4500037.0,48.0
7194,The Adventure of the Copper Beeches,4462995.0,8.0
8370,The Adventure of the Beryl Coronet,4437892.0,8.0
26806,Rag Lalit,4423000.0,23.0


In [12]:
con.close()

# Duplicate Check Summary

This notebook examined different types of duplicates and duplicate-like records within the Spotify dataset.

The initial checks showed that the dataset is technically well structured. No exact duplicate rows were found in any table, and all primary identifier columns were unique. This indicates that the data collection and integration process did not introduce duplicated records at the database level.

However, duplicate analysis at the content level revealed a different picture. More than 8,600 track names appeared multiple times, and over 25,000 track records were associated with repeated titles. While many of these cases represent different songs sharing common names, further investigation showed that a substantial number of tracks correspond to multiple versions of the same musical work.

The version analysis identified thousands of tracks labelled as remasters, remixes, live recordings, radio edits, acoustic versions, deluxe editions, and similar variants. These versions represent the same underlying song but appear as separate records in the dataset. Such entries can artificially increase the importance of certain songs and artists during exploratory analysis and recommendation modelling.

A more detailed duplicate audit based on artist-track combinations revealed nearly 3,900 groups of potential duplicate songs involving more than 8,500 track records. Many of these tracks had almost identical durations and metadata while appearing under different Spotify IDs, albums, or releases. This strongly suggests the presence of re-releases, compilation albums, remasters, and duplicate catalog entries.

The duration-based audit uncovered an additional data quality issue. The dataset contains a considerable amount of non-musical content, including audiobooks, educational lectures, spoken-word recordings, children's stories, radio dramas, ambient sound recordings, and white-noise tracks. More than 1,100 tracks exceeded 15 minutes in duration, and manual inspection confirmed that many of these records do not represent conventional music tracks.

Overall, the duplicate audit demonstrates that the dataset is technically clean but contains several forms of content-level duplication and non-musical material. These findings provide important guidance for the subsequent cleaning phase. In particular, future preprocessing should focus on:

- Removing export artifacts and unnecessary columns.
- Investigating duplicate song versions and re-releases.
- Identifying and handling non-musical content.
- Defining rules for remasters, live recordings, remixes, and other alternative versions.
- Reducing duplicate-like records that may bias exploratory analysis and recommendation models.

The results of this notebook will be used directly in the data cleaning stage before continuing with deeper exploratory analysis and recommendation development.